# 01 · Collect compiling Program.cs files

**Goal:** build a set of real, modern ASP.NET Core `Program.cs` files that compile on their own.
These are the *correct* versions; in the next notebook we break them on purpose to create training data.

**Source:** Microsoft's AspNetCore.Docs repository (`samples` folders, MIT-licensed code).

## 1. Setup
Make the project folder importable and set the filters.

In [1]:
# Cell 3: Setup.
import sys                                              # To add the project folder to Python's import path
from pathlib import Path                                # File paths that work on any OS

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # Project folder, wherever it lives
sys.path.insert(0, str(ROOT))                           # So "import csharp" / "import variants" work

import pandas as pd                                     # Tables
from collections import Counter                         # Count error codes
from csharp import compile_code, compile_many           # Our build tool
from variants import expand_variants                    # Split #if-switched files into clean variants

REPOS = [ROOT / "data" / "raw" / "AspNetCore.Docs",     # Repo 1: the docs (samples and snippets)
         ROOT / "data" / "raw" / "AspNetCore.Docs.Samples"]   # Repo 2: the extra samples repo
P3_EVALS = ROOT.parent / "aspnetcore-coding-agents" / "evals"   # Project 3 results (optional: skipped if not found)
SEEDS_FILE = ROOT / "data" / "processed" / "seeds.jsonl"   # Output: the compiling programs
MIN_CHARS, MAX_CHARS = 150, 2_500                       # Skip tiny programs and programs too long for a small model
for folder in REPOS + [P3_EVALS]:
    print(f"{folder.name:<25} exists: {folder.exists()}")   # All three should say True

AspNetCore.Docs           exists: True
AspNetCore.Docs.Samples   exists: True
evals                     exists: True


## 2. Find and filter the candidates
Keep files that use modern hosting (`WebApplication.CreateBuilder`), are a reasonable length, and are not duplicates.

In [2]:
# Cell 5: Collect programs from both repos (expanded into variants) and from Project 3, then filter.
rows, unreadable = [], 0
for repo in REPOS:                                      # Both repositories
    for p in repo.rglob("*.cs"):                        # Every C# file in any subfolder
        try:
            text = p.read_text(encoding="utf-8-sig")    # utf-8-sig also removes the invisible BOM some files start with
        except (OSError, UnicodeDecodeError):           # Very long paths or odd encodings
            unreadable += 1
            continue
        if "WebApplication.CreateBuilder" not in text:  # Only files that build a web app (.NET 6+ minimal hosting)
            continue
        path = f"{repo.name}/{p.relative_to(repo).as_posix()}"   # Where it came from
        for i, variant in enumerate(expand_variants(text)):      # One clean program per #if variant
            rows.append({"path": f"{path}#v{i}", "group": path, "code": variant})   # group = the source file
print(f"Docs app files: {len({r['group'] for r in rows}):,} -> {len(rows):,} variants (unreadable: {unreadable})")

p3_count = 0
for f in sorted(P3_EVALS.glob("results_*.jsonl")) if P3_EVALS.exists() else []:   # Project 3 result files
    for r in pd.read_json(f, lines=True).to_dict("records"):    # One row per task
        if r["build_ok"] and r["code"]:                         # Only code that compiled
            rows.append({"path": f"p3/{f.stem}/{r['id']}", "group": f"p3/{r['id']}", "code": r["code"]})   # group = the task
            p3_count += 1
print(f"Project 3 programs: {p3_count}")

df = pd.DataFrame(rows)                                 # One row per program
df = df[df["code"].str.contains("WebApplication.CreateBuilder", regex=False)]   # A variant may have lost the app code
df["chars"] = df["code"].str.len()                      # Program length in characters
sized = df[df["chars"].between(MIN_CHARS, MAX_CHARS)].copy()   # Not too short, not too long
print(f"Right length:      {len(sized):,}")
sized["norm"] = sized["code"].str.replace(r"\s+", " ", regex=True).str.strip()   # Same code, ignoring spacing
candidates = sized.drop_duplicates("norm").reset_index(drop=True)   # Remove copies across versions, repos and variants
print(f"Unique candidates: {len(candidates):,} from {candidates['group'].nunique():,} source files/tasks")

Docs app files: 423 -> 1,057 variants (unreadable: 0)
Project 3 programs: 38
Right length:      906
Unique candidates: 769 from 336 source files/tasks


## 3. Test the build tool
Compile one correct and one broken program to check that the sandbox works.

In [3]:
# Cell 7: Sanity check (the first run creates the sandbox and takes ~10-20 s).
good = 'var app = WebApplication.CreateBuilder(args).Build();\napp.MapGet("/", () => "Hi");\napp.Run();'   # Correct
bad = good.replace('"Hi");', '"Hi")')                   # Same code with a missing semicolon
r1, r2 = compile_code(good), compile_code(bad)          # Build both
print("good:", r1.succeeded, f"{r1.seconds} s")         # Expect True
print("bad: ", r2.succeeded, f"{r2.seconds} s")         # Expect False
print(r2.error_text(bad))                               # Expect CS1002: ; expected

good: True 2.8 s
bad:  False 2.1 s
Line 2: error CS1002: ; expected
    > app.MapGet("/", () => "Hi")


## 4. Compile every candidate
Each one is built in a sandbox (4 in parallel). Only programs that compile on their own are kept.

In [4]:
# Cell 9: Compile all candidates in parallel and keep the ones that build.
results = compile_many(candidates["code"].tolist(), workers=4)   # One BuildResult per candidate, same order
candidates["compiles"] = [r.succeeded for r in results]  # True/False per candidate
seeds = candidates[candidates["compiles"]].reset_index(drop=True)   # Keep only the compiling ones
seeds.insert(0, "id", [f"seed_{i:04d}" for i in range(len(seeds))])   # Stable IDs: seed_0000, seed_0001, ...

SEEDS_FILE.parent.mkdir(parents=True, exist_ok=True)    # Make sure data\processed exists
seeds[["id", "group", "path", "code", "chars"]].to_json(SEEDS_FILE, orient="records", lines=True, force_ascii=False)
print(f"Compiled: {len(seeds):,} of {len(candidates):,} ({len(seeds) / len(candidates):.0%})")
print(f"From {seeds['group'].nunique():,} different source files/tasks")   # What matters for an honest test split
print("Saved:", SEEDS_FILE)

100%|██████████| 769/769 [13:13<00:00,  1.03s/it]

Compiled: 367 of 769 (48%)
From 127 different source files/tasks
Saved: d:\AI Projects\csharp-fix-lora\data\processed\seeds.jsonl


## 5. Look at the seeds
How long are they, and what does a typical one look like?

In [5]:
# Cell 11: Length summary and one example.
print(seeds["chars"].describe().round(0))               # Min / mean / max length in characters
print("\nExample:", seeds.loc[0, "path"])               # Where the first seed came from
print(seeds.loc[0, "code"])                             # Its code

count     367.0
mean      708.0
std       391.0
min       152.0
25%       445.0
50%       617.0
75%       874.0
max      2472.0
Name: chars, dtype: float64

Example: AspNetCore.Docs/aspnetcore/fundamentals/apis/APIWithControllers/Program.cs#v0
namespace APIWithControllers;

public class Program
{
    public static void Main(string[] args)
    {
        var builder = WebApplication.CreateBuilder(args);

        builder.Services.AddControllers();
        var app = builder.Build();

        app.UseHttpsRedirection();

        app.MapControllers();

        app.Run();
    }
}



## 6. Why candidates fail to compile
The most common first error among candidates that didn't build. This tells us whether we're losing them for a fixable reason.

In [6]:
# Cell 13: Count the first compiler error of every candidate that failed.
failed = [r for r in results if not r.succeeded]        # BuildResults of the candidates that didn't compile
first_codes = Counter(r.errors[0]["code"] if r.errors else "none" for r in failed)   # First error code of each
print(f"Failed: {len(failed):,}")
for code, n in first_codes.most_common(10):             # The 10 most common
    print(f"  {code:<8} {n:>5}")

Failed: 402
  CS0234     189
  CS0246     156
  CS1061      46
  CS0103       8
  CS1529       1
  CS1593       1
  CS1069       1
